In [ ]:
# # 九曜：天墟 LF01 — 旁白／蕭曜霖 國漫配音版（Qwen3-TTS，Kaggle）2026-09-30
#
# 咖哩試聽：BreezyVoice／OpenAI 台灣國語版「氣勢、音色不對」，要的是**大陸國漫配音**的感覺。
# → 改用 Qwen3-TTS VoiceDesign，提示詞寫國漫史詩旁白／強者教官、標準普通話播音腔（正面描述，不寫「不要…」；
#   v2 寫「不要大陸普通話腔調」才會偏成粵語）。每角色 3 個候選聲音 × 每句 2 次。
# 由 Claude 經 Kaggle API 推送執行；MP3 以 base64 印進執行紀錄供 Claude 取回。


In [ ]:
import subprocess
print(subprocess.run("pip install -q -U qwen-tts soundfile 2>&1 | tail -3; nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)


In [ ]:
import json, os
VOICES = json.loads(r'''{"VO": {"name": "旁白", "ref_text": "天地初開，九曜懸空。千年之後，一座古老的學院，依然矗立在群山之巔，等待著下一個踏入其中的少年。", "prompt": "中國國漫（3D玄幻動畫）片頭的史詩旁白，成熟男聲約四十歲，渾厚低沉、胸腔共鳴強，音色有磁性和顆粒感。字正腔圓的標準普通話播音腔。語速緩慢，每個詞都有分量，停頓拉長，語氣莊重、宏大、蒼茫而神秘，像在講述一個古老世界的開端。"}, "C05": {"name": "蕭曜霖", "ref_text": "所有人給我聽好。在這裡，沒有人會等你。明天一早，演武場集合，遲到者，後果自負。", "prompt": "中國國漫（3D玄幻動畫）裡的強者教官，四十歲左右的男聲，低沉渾厚、略帶沙啞顆粒感，氣場極強。字正腔圓的標準普通話，專業配音演員的表演腔。語氣冷峻、不容置疑，一字一頓，壓迫感十足，像站在高處俯視新生下達命令。"}}''')
LINES = [["VO", "LF01-A01_旁白", "九曜界……天玄院。", 4], ["C05", "LF01-A10a_蕭曜霖", "不用等明天。", 3], ["C05", "LF01-A10b_蕭曜霖", "半刻鐘後，新生演武場集合。", 4]]
CANDS, TAKES = 3, 2
OUT = "/kaggle/working/voice_guoman"; os.makedirs(f"{OUT}/ref", exist_ok=True); os.makedirs(f"{OUT}/clone", exist_ok=True)


In [ ]:
# 設計候選參考聲音（VoiceDesign）；T4 → float32＋sdpa
import torch, soundfile as sf, gc
from qwen_tts import Qwen3TTSModel
DTYPE = torch.float32
design = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", device_map="cuda:0", dtype=DTYPE, attn_implementation="sdpa")
REFS = {}
for vid, v in VOICES.items():
    for c in range(1, CANDS + 1):
        torch.manual_seed(1000 * c + len(vid)); torch.cuda.manual_seed_all(1000 * c + len(vid))
        wavs, sr = design.generate_voice_design(text=v["ref_text"], language="Chinese", instruct=v["prompt"])
        w = wavs[0][: int(sr * 20)]
        sf.write(f"{OUT}/ref/{vid}_{v['name']}_c{c}.wav", w, sr); REFS[(vid, c)] = (w, sr, v["ref_text"])
        print("ref", vid, v["name"], f"c{c}", f"{len(wavs[0])/sr:.1f}s")
del design; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# 每個候選聲音唸每句台詞 TAKES 次（Base 模型 clone）；限制長度防失控
clone = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-Base", device_map="cuda:0", dtype=DTYPE, attn_implementation="sdpa")
PROMPTS = {key: clone.create_voice_clone_prompt(ref_audio=(w, sr), ref_text=t) for key, (w, sr, t) in REFS.items()}
def say(text, prompt, cut):
    try:
        return clone.generate_voice_clone(text=text, language="Chinese", voice_clone_prompt=prompt, max_new_tokens=int(12 * max(8, 3 * cut)))
    except TypeError:
        return clone.generate_voice_clone(text=text, language="Chinese", voice_clone_prompt=prompt)
for spk, name, text, cut in LINES:
    for c in range(1, CANDS + 1):
        for k in range(1, TAKES + 1):
            wavs, sr = say(text, PROMPTS[(spk, c)], cut)
            w = wavs[0][: int(sr * max(8, 3 * cut))]
            sf.write(f"{OUT}/clone/{name}_c{c}_t{k}.wav", w, sr)
            print(name, f"c{c} t{k}", f"{len(wavs[0])/sr:.2f}s", text)


In [ ]:
# 壓 MP3，並以 base64 印進執行紀錄（Claude 從雲端抓不到 Output 檔）
import glob, base64, shutil
dst = "/kaggle/working/voice_guoman_tiny"; os.makedirs(dst, exist_ok=True)
for w in sorted(glob.glob(f"{OUT}/**/*.wav", recursive=True)):
    m = f"{dst}/{os.path.basename(w)[:-4]}.mp3"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", w, "-t", "20", "-ac", "1", "-b:a", "64k", m], check=True)
shutil.make_archive("/kaggle/working/LF01_voice_guoman_tiny", "zip", dst)
for m in sorted(glob.glob(f"{dst}/*.mp3")):
    print(f"@@MP3 {os.path.basename(m)} {base64.b64encode(open(m, 'rb').read()).decode()}")
